# Speech-to-Speech Translation — Whisper + Helsinki-NLP + MMS-TTS

Translating spoken language into spoken language in another tongue requires solving
three distinct problems in sequence: recognizing what was said (ASR), converting the
words to the target language (MT), and synthesizing speech from the translation (TTS).
This notebook shows how to chain three specialist models — one for each stage — into a
coherent pipeline, and why each specialist beats a generalist at its own task.

| Step | Concept | Key Idea |
|------|---------|---------|
| 1 | Why the chain is hard | Dictionary lookup fails on fluency; cascading errors compound |
| 2 | Three specialists | Whisper (ASR) + Helsinki-NLP (MT) + MMS-TTS (TTS) |
| 3 | Pipeline implementation | Modular functions; clear data contracts between stages |
| 4 | Live demo | Translate an English sentence to Spanish |
| 5 | Your turn | Change the target language model |
| Summary | Key insights | Consolidated takeaways |


In [ ]:
# ── Install dependencies (skip if already present) ─────────────────────────────
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

_ensure("transformers")
_ensure("torch")
_ensure("soundfile")
_ensure("numpy")
print("Dependencies ready.")


In [ ]:
# ── Imports and deterministic seed ─────────────────────────────────────────────
import torch
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Torch version : {torch.__version__}")
print(f"Device        : {'cuda' if torch.cuda.is_available() else 'cpu'}")


---

## Part 1 — Why does word-by-word translation fail?

The most obvious translation algorithm is a dictionary: for each word in the source
sentence, look up its equivalent in the target language and concatenate the results.
This breaks immediately on anything that requires reordering, agreement, or idioms.

Predict before you run:
> English: "I am going to the bank to get some money."
> Spanish word order differs from English, and "bank" is ambiguous (financial vs. river).
>
> What does a word-by-word English→Spanish dictionary produce?
> Will it read naturally to a native Spanish speaker?

Run the cell below to see the failure.


In [ ]:
# ── Naive word-by-word dictionary translation ──────────────────────────────────
DICT_EN_ES = {
    "i": "yo", "am": "estoy", "going": "yendo", "to": "a",
    "the": "el", "bank": "banco", "get": "obtener", "some": "algo",
    "money": "dinero", "i'm": "estoy", "hungry": "hambriento",
    "she": "ella", "has": "tiene", "been": "sido", "reading": "leyendo",
    "book": "libro", "for": "por", "an": "una", "hour": "hora",
}

def word_by_word(sentence: str) -> str:
    tokens = sentence.lower().replace(".", "").split()
    translated = [DICT_EN_ES.get(t, f"[{t}?]") for t in tokens]
    return " ".join(translated)

sentences = [
    "I am going to the bank to get some money.",
    "She has been reading a book for an hour.",
    "It is raining cats and dogs.",
]

print("Word-by-word dictionary translation:")
for s in sentences:
    t = word_by_word(s)
    print(f"  EN: {s}")
    print(f"  ES: {t}")
    print()

print("  -> Problems:")
print("  -> 1. Word order not adjusted (English SVO != Spanish SVO but with reordering in idioms).")
print("  -> 2. Unknown words left as [?] (cats, dogs, raining).")
print("  -> 3. Idioms like 'raining cats and dogs' have no word-level equivalent.")
print("  -> 4. Inflection/agreement not handled (el banco vs la banca).")


#### What just happened — and what's missing

Word-by-word translation fails on four levels:
1. **Word order** — many language pairs require structural reordering (SVO → SOV, etc.).
2. **Unknown words** — OOV tokens produce garbage output.
3. **Idioms** — compositional meaning is not the sum of word meanings.
4. **Morphological agreement** — gender, number, tense inflections are not in a dictionary.

A neural seq2seq model trained on millions of parallel sentences learns all four
implicitly. That is the Helsinki-NLP opus-mt family.


---

## Part 2 — Three specialists: why one model can't do everything well

The pipeline uses three models, each trained on a different task and data distribution:

### 2a. Whisper — Automatic Speech Recognition

Whisper is a seq2seq encoder-decoder trained on 680 k hours of web audio.

**CTC-free transcript probability:**

$$P(W \mid X) = \prod_{t=1}^{T} P(w_t \mid w_{<t},\, X)$$

Plain-English gloss: the decoder generates word tokens one at a time, conditioning on
all prior words *and* on the full audio encoding $X$. The audio encoder uses 2D
convolutions to build log-Mel spectrogram features, then a transformer to contextualize them.

### 2b. Helsinki-NLP opus-mt — Neural Machine Translation

opus-mt models are compact (74 M parameters) MarianMT encoder-decoders, trained on
OPUS parallel corpora for 1,000+ language pairs.

$$P(Y \mid X_{\text{text}}) = \prod_{j=1}^{|Y|} P(y_j \mid y_{<j},\, \text{Enc}(X_{\text{text}}))$$

Plain-English gloss: same seq2seq formula as Whisper, but the input is tokenized text
(not audio), and the training signal is aligned bilingual sentence pairs.

### 2c. MMS-TTS — Text-to-Speech

Meta's Massively Multilingual Speech TTS covers 1,100+ languages using a VITS
architecture (flow-based vocoder conditioned on phoneme embeddings).

| Stage | Model | Input | Output | Params |
|-------|-------|-------|--------|--------|
| 1 – ASR | Whisper-base | audio waveform | transcript text | 74 M |
| 2 – MT  | opus-mt-en-es | source text    | target text     | 74 M |
| 3 – TTS | MMS-TTS-spa   | target text    | audio waveform  | ~82 M |


In [ ]:
# ── Load ASR: Whisper-base ──────────────────────────────────────────────────────
from transformers import pipeline as hf_pipeline

print("Loading Whisper-base for speech recognition ...")
asr = hf_pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base",
    device=-1,   # CPU
)
print(f"  -> Whisper-base loaded  (model: {asr.model.__class__.__name__})")


In [ ]:
# ── Load MT: Helsinki-NLP opus-mt-en-es ─────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_MODEL_NAME = "Helsinki-NLP/opus-mt-en-es"
print(f"Loading {MT_MODEL_NAME} ...")
mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_NAME)
mt_model     = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_NAME)
mt_model.eval()

print(f"  -> MT model loaded  (encoder layers: {mt_model.config.encoder_layers}, "
      f"d_model: {mt_model.config.d_model})")


In [ ]:
# ── (Optional) Load TTS: facebook/mms-tts-spa ───────────────────────────────────
# TTS requires additional system libraries (libsndfile).
# Wrapped in try/except so the notebook still runs if TTS is unavailable.

tts_available = False
tts = None
try:
    tts = hf_pipeline(
        "text-to-speech",
        model="facebook/mms-tts-spa",
        device=-1,
    )
    tts_available = True
    print("  -> MMS-TTS-spa loaded.")
except Exception as e:
    print(f"  -> TTS not available ({type(e).__name__}: {e})")
    print("     The pipeline will run ASR + MT only; TTS output will be skipped.")


#### What just happened — and what's missing

We loaded the three specialist models independently.
Each model was trained on a different task and dataset; none of them were trained
end-to-end together. The pipeline treats their interfaces as contracts:
- ASR contract: audio file path or array → transcript string.
- MT contract: source-language string → target-language string.
- TTS contract: target-language string → audio array + sample rate.

Cascading errors are the main weakness: an ASR transcription mistake propagates into
the MT input and amplifies in the TTS output. End-to-end speech translation models
(e.g., SeamlessM4T) address this by training all three jointly.


---

## Part 3 — Pipeline implementation

We implement each stage as a pure function with a clear input/output contract,
then compose them in `translate_speech_to_speech()`.


In [ ]:
# ── Stage 2: translate_text ──────────────────────────────────────────────────────
# The MT stage takes a plain string and returns a translated string.
# We can test this without any audio.

def translate_text(source_text: str, max_length: int = 512) -> str:
    """Translate English text to Spanish using Helsinki-NLP/opus-mt-en-es."""
    # Tokenize
    inputs = mt_tokenizer(
        source_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )
    # Generate
    with torch.no_grad():
        output_ids = mt_model.generate(**inputs, max_length=max_length)

    # Decode
    translated = mt_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return translated.strip()

print("translate_text() ready.")


In [ ]:
# ── Stage 3 (optional): synthesize_speech ───────────────────────────────────────
import io
import soundfile as sf

def synthesize_speech(text: str, sample_rate: int = 16000):
    """Convert text to speech. Returns (audio_array, sample_rate) or None if TTS unavailable."""
    if not tts_available:
        print("  -> TTS not available; skipping audio synthesis.")
        return None, sample_rate
    result = tts(text)
    audio  = result["audio"]
    sr     = result.get("sampling_rate", sample_rate)
    return audio, sr


def translate_speech_to_speech(audio_path: str):
    """Full pipeline: audio file -> transcript -> Spanish -> audio (if TTS available)."""
    # Stage 1: ASR
    asr_result  = asr(audio_path)
    transcript  = asr_result["text"]
    print(f"  Stage 1 ASR   : {transcript}")

    # Stage 2: MT
    translation = translate_text(transcript)
    print(f"  Stage 2 MT    : {translation}")

    # Stage 3: TTS
    audio, sr   = synthesize_speech(translation)
    if audio is not None:
        print(f"  Stage 3 TTS   : {len(audio)} samples @ {sr} Hz")
    else:
        print("  Stage 3 TTS   : skipped (TTS unavailable)")

    return {"transcript": transcript, "translation": translation, "audio": audio, "sample_rate": sr}

print("Full pipeline ready.")


#### What just happened — and what's missing

Each stage is isolated behind a function boundary:
- `asr(audio_path)` returns a dict with `"text"` — we extract that string.
- `translate_text(text)` returns a string — we pass it to TTS.
- `synthesize_speech(text)` returns an array — we could write it to WAV.

The data contracts make each stage independently testable.
If the ASR model is swapped for a better one, `translate_text` and `synthesize_speech`
are unaffected — they only depend on the contract (string in, string out).


---

## Part 4 — Live demo: text-only translation (no audio file needed)

We skip the ASR stage and test the MT and TTS stages directly with English text.
This lets us verify the core translation quality without needing a microphone or audio file.


In [ ]:
# ── Live demo: direct text translation ─────────────────────────────────────────

test_sentences = [
    "The weather is beautiful today.",
    "I would like to order a coffee with milk, please.",
    "The train leaves at half past three in the afternoon.",
]

print("English -> Spanish (Helsinki-NLP/opus-mt-en-es):")
print()
for s in test_sentences:
    t = translate_text(s)
    print(f"  EN: {s}")
    print(f"  ES: {t}")
    print()

print("  -> Neural MT handles word order, morphology, and idiom.")
print("  -> Compare to the word-by-word results in Part 1.")


In [ ]:
# ── Your turn — change the target language model ───────────────────────────────
# CHANGE: set TARGET_LANG below and re-run.
#    Helsinki-NLP opus-mt models follow the pattern "opus-mt-{src}-{tgt}".
#    Common language codes: "fr" (French), "de" (German), "it" (Italian),
#                           "pt" (Portuguese), "zh" (Chinese), "ar" (Arabic)

TARGET_LANG = "fr"   # try: "de", "it", "pt"

new_model_name = f"Helsinki-NLP/opus-mt-en-{TARGET_LANG}"
print(f"Loading {new_model_name} ...")
try:
    new_tok = AutoTokenizer.from_pretrained(new_model_name)
    new_mdl = AutoModelForSeq2SeqLM.from_pretrained(new_model_name)
    new_mdl.eval()

    test_text = "The weather is beautiful today."
    inputs    = new_tok(test_text, return_tensors="pt")
    with torch.no_grad():
        out_ids = new_mdl.generate(**inputs, max_length=200)
    result = new_tok.decode(out_ids[0], skip_special_tokens=True)

    print(f"  EN: {test_text}")
    print(f"  {TARGET_LANG.upper()}: {result}")
    print("  -> Swapping the MT model is all that changes — the pipeline contract stays the same.")
except Exception as e:
    print(f"  -> Could not load {new_model_name}: {e}")
    print("     Check that the language code is valid on the HuggingFace hub.")


---

## Summary

| Step | Concept | Key Idea |
|------|---------|---------|
| 1 | Why dictionary translation fails | Word order, morphology, idioms require learned distributions |
| 2 | Three specialists | Whisper (ASR) + opus-mt (MT) + MMS-TTS (TTS) |
| 3 | Pipeline implementation | Clear data contracts; each stage independently testable |
| 4 | Live demo | Neural MT preserves word order and morphological agreement |
| 5 | Your turn | Swapping the MT model name changes the target language |

**Key insights to keep**

- Speech-to-speech translation is naturally decomposed into ASR → MT → TTS; each sub-problem has its own specialist model trained on its own data.
- Cascading errors are the main weakness of pipeline systems: an ASR mistake is amplified by MT and audible in TTS — this motivates end-to-end models like SeamlessM4T.
- Helsinki-NLP opus-mt models (~74 M parameters) trade quality for speed; they run on CPU in seconds, unlike 10 B+ parameter translation models.
- Modular design enables language switching with a single model-name change; the surrounding pipeline code is unaffected.
- Neural MT solves what dictionary lookup cannot: word reordering, morphological inflection, and idiomatic expression.
